# SmartCropAI - Tamil Nadu field-pilot training

This notebook is a reproducible training and evaluation starting point, not a claim that the model already works. The provisional pilot crops are rice, tomato, and banana; agree the supported crop/disease labels with local farmers and an agricultural expert before collecting labels. The notebook trains only on the classes present in a verified manifest. It does not recognize every plant.

The repository currently has only 10 raw sample images and a recorded 10% score on 10 test samples. Do not use those artifacts for diagnosis. [PlantVillage](https://huggingface.co/datasets/mohanty/PlantVillage) provides a large controlled-image benchmark; [PlantDoc](https://github.com/pratikkayal/PlantDoc-Dataset) provides a smaller, more varied image set under CC BY 4.0. Check each dataset's license and exact class definitions before using it. Neither public benchmark substitutes for an independent Tamil Nadu field test, and any crop/disease gaps need locally collected photos.

## Prepare the Drive data

Create `MyDrive/smartcropai_pilot/dataset/images/` and place consented, metadata-stripped photos beneath it. Add `manifest.csv` in `dataset/` with these columns:

| Column | Meaning |
| --- | --- |
| `relative_path` | Path under `images/`, e.g. `rice/healthy/photo001.jpg` |
| `label` | Exact crop-and-condition label agreed with an expert, e.g. `Rice___Healthy` |
| `group_id` | Pseudonymous farm/collection-group ID; keep every group's photos in exactly one split |
| `split` | `train`, `val`, `test`, or optional `field_test` |
| `source` | `tamil_nadu_field` or the documented public dataset name |
| `label_status` | Must be `expert_verified` for this pilot workflow |
| `consent_or_license` | Consent record reference or applicable public-dataset license |

Use distinct farms/groups for train, validation, and test. Keep `field_test` from farms and phones not used for training. Do not put names, phone numbers, precise farm locations, or other personal information in the manifest. The split is checked for group and exact-file leakage. Aim for at least 100 varied photos per class from at least 5 independent farm groups; smaller data may run, but is only exploratory. Include healthy, diseased, different severities, lighting, backgrounds, and phone cameras. Do not guess disease labels: mark uncertain images for expert review and leave them out.

## Run

In Colab select **Runtime > Change runtime type > T4 GPU** (or another available GPU), then run cells in order. Outputs are written to Drive. Keep the test set untouched until the final evaluation. The reported Colab latency is not Android latency; benchmark on the actual target phones before deployment.

In [ ]:
%pip -q install scikit-learn pandas pillow
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json
import hashlib
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score

DATASET_DIR = Path('/content/drive/MyDrive/smartcropai_pilot/dataset')
IMAGE_DIR = DATASET_DIR / 'images'
MANIFEST_PATH = DATASET_DIR / 'manifest.csv'
ARTIFACT_DIR = Path('/content/drive/MyDrive/smartcropai_pilot/runs') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 10
FINE_TUNE_EPOCHS = 10
SEED = 42
tf.keras.utils.set_random_seed(SEED)
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Manifest:', MANIFEST_PATH)
print('Run output:', ARTIFACT_DIR)

In [ ]:
required = {'relative_path', 'label', 'group_id', 'split', 'source', 'label_status', 'consent_or_license'}
if not MANIFEST_PATH.is_file():
    raise FileNotFoundError(f'Add the verified image manifest first: {MANIFEST_PATH}')
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Add image files under: {IMAGE_DIR}')

manifest = pd.read_csv(MANIFEST_PATH, dtype=str).fillna('').apply(lambda col: col.str.strip())
missing = required - set(manifest.columns)
if missing:
    raise ValueError(f'Manifest is missing required columns: {sorted(missing)}')
if manifest.empty or (manifest[list(required)] == '').any().any():
    raise ValueError('Manifest has no rows or has blank required values.')
if not manifest['label_status'].eq('expert_verified').all():
    raise ValueError('Only expert_verified rows may be used. Resolve uncertain labels before training.')
allowed_splits = {'train', 'val', 'test', 'field_test'}
if not set(manifest['split']).issubset(allowed_splits):
    raise ValueError(f'Split must be one of {sorted(allowed_splits)}.')
if not {'train', 'val', 'test'}.issubset(set(manifest['split'])):
    raise ValueError('Manifest needs separate train, val, and test rows.')

def resolve_image(relative_path):
    path = (IMAGE_DIR / relative_path).resolve()
    if not path.is_relative_to(IMAGE_DIR.resolve()):
        raise ValueError(f'Image path escapes images directory: {relative_path}')
    if not path.is_file():
        raise FileNotFoundError(path)
    return str(path)

manifest['image_path'] = manifest['relative_path'].map(resolve_image)
group_splits = manifest.groupby('group_id')['split'].nunique()
if (group_splits > 1).any():
    leaking = group_splits[group_splits > 1].index.tolist()[:10]
    raise ValueError(f'Farm/collection groups appear in multiple splits: {leaking}')

digest_splits = {}
for row in manifest.itertuples(index=False):
    digest = hashlib.sha256(Path(row.image_path).read_bytes()).hexdigest()
    digest_splits.setdefault(digest, set()).add(row.split)
leaking_hashes = [digest for digest, splits in digest_splits.items() if len(splits) > 1]
if leaking_hashes:
    raise ValueError(f'Found {len(leaking_hashes)} exact duplicate images across splits; fix the split before training.')

classes = sorted(manifest['label'].unique().tolist())
if len(classes) < 2:
    raise ValueError('At least two verified classes are required.')
split_counts = manifest.groupby(['split', 'label']).size().unstack(fill_value=0)
for split in ('train', 'val', 'test'):
    absent = sorted(set(classes) - set(split_counts.loc[split][split_counts.loc[split] > 0].index))
    if absent:
        raise ValueError(f'{split} is missing classes {absent}; do not evaluate on an incomplete split.')
print('Images per split and class:')
display(split_counts)
print('Unique farm/collection groups per split:')
display(manifest.groupby('split')['group_id'].nunique())
print('Classes:', classes)
print('Field-test rows:', int((manifest['split'] == 'field_test').sum()))

## Train a compact transfer-learning baseline

Images are decoded as RGB and resized consistently. Augmentation is part of the training model and is disabled for evaluation/inference. The MobileNetV2 preprocessing layer is inside the exported model, so app preprocessing must pass RGB pixel values in the `[0, 255]` range without applying another normalization.

In [ ]:
class_to_index = {label: index for index, label in enumerate(classes)}
index_to_class = {index: label for label, index in class_to_index.items()}
AUTOTUNE = tf.data.AUTOTUNE

def make_dataset(frame, shuffle=False):
    paths = frame['image_path'].astype(str).to_numpy()
    labels = frame['label'].map(class_to_index).to_numpy(dtype=np.int32)
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(len(frame), seed=SEED, reshuffle_each_iteration=True)
    def load_image(path, label):
        encoded = tf.io.read_file(path)
        image = tf.io.decode_image(encoded, channels=3, expand_animations=False)
        image.set_shape([None, None, 3])
        image = tf.image.resize(image, IMAGE_SIZE, antialias=True)
        return tf.cast(image, tf.float32), label
    return dataset.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_df = manifest[manifest['split'] == 'train'].copy()
val_df = manifest[manifest['split'] == 'val'].copy()
test_df = manifest[manifest['split'] == 'test'].copy()
field_test_df = manifest[manifest['split'] == 'field_test'].copy()
train_ds = make_dataset(train_df, shuffle=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)

augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomContrast(0.1),
], name='training_augmentation')
inputs = tf.keras.Input(shape=(*IMAGE_SIZE, 3), name='rgb_image_0_to_255')
x = augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)
backbone = tf.keras.applications.MobileNetV2(include_top=False, weights='imagenet', input_shape=(*IMAGE_SIZE, 3))
backbone.trainable = False
x = backbone(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.25)(x)
outputs = tf.keras.layers.Dense(len(classes), activation='softmax', name='class_probabilities')(x)
model = tf.keras.Model(inputs, outputs, name='smartcropai_tamil_nadu_pilot')
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6),
]
history_head = model.fit(train_ds, validation_data=val_ds, epochs=INITIAL_EPOCHS, callbacks=callbacks)

backbone.trainable = True
for layer in backbone.layers[:-30]:
    layer.trainable = False
for layer in backbone.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history_fine = model.fit(train_ds, validation_data=val_ds, epochs=FINE_TUNE_EPOCHS, callbacks=callbacks)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
keras_path = ARTIFACT_DIR / 'pilot_model.keras'
model.save(keras_path)
training_history = {
    metric: list(history_head.history.get(metric, [])) + list(history_fine.history.get(metric, []))
    for metric in sorted(set(history_head.history) | set(history_fine.history))
}
(ARTIFACT_DIR / 'training_history.json').write_text(json.dumps(training_history, indent=2), encoding='utf-8')
print('Saved:', keras_path)

## Evaluate, quantize, and save a model card

Validation is for training decisions; `test` and `field_test` remain held out. Full integer quantization can fail when an operation is unsupported; do not silently fall back to a different format. The notebook measures quantized accuracy and quantization agreement on the held-out images. Agreement with the float model is not the same as correctness. Colab CPU timing is only a rough reference, never a substitute for measuring on Android.

In [ ]:
def evaluate_keras(frame, split_name):
    if frame.empty:
        return None
    dataset = make_dataset(frame)
    y_true = frame['label'].map(class_to_index).to_numpy(dtype=np.int32)
    probabilities = model.predict(dataset, verbose=0)
    y_pred = probabilities.argmax(axis=1)
    return {
        'split': split_name,
        'samples': int(len(frame)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'classification_report': classification_report(y_true, y_pred, labels=list(range(len(classes))), target_names=classes, output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(len(classes)))).tolist(),
    }

keras_results = [evaluate_keras(test_df, 'test')]
if not field_test_df.empty:
    keras_results.append(evaluate_keras(field_test_df, 'field_test'))
print('Float model held-out results:')
for result in keras_results:
    print(result['split'], 'accuracy=', round(result['accuracy'], 4), 'macro-F1=', round(result['macro_f1'], 4), 'samples=', result['samples'])

def representative_data():
    sample = train_df.groupby('label', group_keys=False).head(100)
    for row in sample.itertuples(index=False):
        with Image.open(row.image_path) as source:
            image = source.convert('RGB').resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            yield [np.expand_dims(np.asarray(image, dtype=np.float32), axis=0)]

converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8
tflite_model = converter.convert()
tflite_path = ARTIFACT_DIR / 'pilot_model_int8.tflite'
tflite_path.write_bytes(tflite_model)
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()
input_info = interpreter.get_input_details()[0]
output_info = interpreter.get_output_details()[0]
if input_info['dtype'] != np.int8 or output_info['dtype'] != np.int8:
    raise RuntimeError('Export is not full integer INT8; refusing to label it as an INT8 model.')
input_scale, input_zero = input_info['quantization']
output_scale, output_zero = output_info['quantization']
if input_scale <= 0 or output_scale <= 0:
    raise RuntimeError('TFLite model has invalid quantization scales.')

def evaluate_tflite(frame, split_name):
    if frame.empty:
        return None
    y_true, y_pred, confidences, latencies = [], [], [], []
    for row in frame.itertuples(index=False):
        with Image.open(row.image_path) as source:
            image = source.convert('RGB').resize(IMAGE_SIZE, Image.Resampling.BILINEAR)
            rgb = np.asarray(image, dtype=np.float32)
        quantized = np.clip(np.rint(rgb / input_scale + input_zero), -128, 127).astype(np.int8)[None, ...]
        interpreter.set_tensor(input_info['index'], quantized)
        start = __import__('time').perf_counter()
        interpreter.invoke()
        latencies.append((__import__('time').perf_counter() - start) * 1000)
        raw = interpreter.get_tensor(output_info['index'])[0].astype(np.float32)
        probabilities = (raw - output_zero) * output_scale
        y_true.append(class_to_index[row.label])
        y_pred.append(int(probabilities.argmax()))
        confidences.append(float(probabilities.max()))
    return {
        'split': split_name,
        'samples': int(len(frame)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0)),
        'classification_report': classification_report(y_true, y_pred, labels=list(range(len(classes))), target_names=classes, output_dict=True, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred, labels=list(range(len(classes)))).tolist(),
        'mean_max_probability': float(np.mean(confidences)),
        'mean_invoke_latency_ms_colab': float(np.mean(latencies)),
        'model_size_bytes': int(tflite_path.stat().st_size),
    }

tflite_results = [evaluate_tflite(test_df, 'test')]
if not field_test_df.empty:
    tflite_results.append(evaluate_tflite(field_test_df, 'field_test'))
for result in tflite_results:
    print(result['split'], 'accuracy=', round(result['accuracy'], 4), 'macro-F1=', round(result['macro_f1'], 4), 'samples=', result['samples'], 'INT8 size MB=', round(result['model_size_bytes'] / 1_000_000, 2), 'Colab invoke ms=', round(result['mean_invoke_latency_ms_colab'], 2))

run_metadata = {
    'project': 'SmartCropAI Tamil Nadu pilot',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'status': 'experimental_not_field_validated' if field_test_df.empty else 'experimental_field_tested_review_required',
    'tensorflow_version': tf.__version__,
    'image_size': list(IMAGE_SIZE),
    'input': 'RGB uint8 pixels resized to 224x224; model includes MobileNetV2 preprocessing',
    'class_order': classes,
    'class_to_index': class_to_index,
    'confidence_threshold': None,
    'confidence_threshold_note': 'Set only after reviewing per-class validation and independent field-test results; no diagnosis threshold is certified.',
    'group_split_counts': manifest.groupby('split')['group_id'].nunique().to_dict(),
    'image_counts': manifest.groupby('split').size().to_dict(),
    'keras_metrics': keras_results,
    'int8_metrics': tflite_results,
    'int8_input_quantization': {'scale': float(input_scale), 'zero_point': int(input_zero)},
    'int8_output_quantization': {'scale': float(output_scale), 'zero_point': int(output_zero)},
    'int8_size_bytes': int(tflite_path.stat().st_size),
}
(ARTIFACT_DIR / 'class_labels.json').write_text(json.dumps({'classes': classes, 'class_to_index': class_to_index}, indent=2), encoding='utf-8')
(ARTIFACT_DIR / 'evaluation_and_model_card.json').write_text(json.dumps(run_metadata, indent=2, default=float), encoding='utf-8')
print('Saved model, class labels, and evaluation to:', ARTIFACT_DIR)
print('Do not deploy until an agricultural expert reviews labels, field-test results, and per-class errors.')